## Useful links

[Ref 1](http://www.chokkan.org/software/crfsuite/tutorial.html)

[Ref 2](https://github.com/TeamHG-Memex/sklearn-crfsuite/blob/master/docs/CoNLL2002.ipynb)



## Install CRF

In [ ]:
!pip3 install -U 'scikit-learn<0.24'
!pip3 install sklearn-crfsuite
#!pip3 install pystruct

## Imports

In [ ]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV

import sklearn_crfsuite
from sklearn_crfsuite import scorers
from sklearn_crfsuite import metrics
from collections import defaultdict, Counter
import os
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import accuracy_score
from itertools import chain
import pandas as pd
import re
import numpy as np
import scipy.stats
from gensim.models import Word2Vec
# from keras.utils import to_categorical
# from pystruct.models import GraphCRF, LatentNodeCRF
# from pystruct.learners import NSlackSSVM, OneSlackSSVM, LatentSSVM

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from collections import Counter

## Read values from zip file

In [ ]:
!wget https://dl.dropboxusercontent.com/s/fwka1oj44f6l40b/user_files.zip

!unzip user_files.zip

!ls
DATA_PATH = '/content/user_files/'

--2022-06-10 14:52:00--  https://dl.dropboxusercontent.com/s/fwka1oj44f6l40b/user_files.zip
Resolving dl.dropboxusercontent.com (dl.dropboxusercontent.com)... 162.125.7.15, 2620:100:601c:15::a27d:60f
Connecting to dl.dropboxusercontent.com (dl.dropboxusercontent.com)|162.125.7.15|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 108376 (106K) [application/zip]
Saving to: ‘user_files.zip’

user_files.zip      100%[===================>] 105.84K   602KB/s    in 0.2s    

2022-06-10 14:52:01 (602 KB/s) - ‘user_files.zip’ saved [108376/108376]

Archive:  user_files.zip
   creating: user_files/
  inflating: user_files/user_1.csv   
  inflating: user_files/user_10.csv  
  inflating: user_files/user_11.csv  
  inflating: user_files/user_12.csv  
  inflating: user_files/user_13.csv  
  inflating: user_files/user_14.csv  
  inflating: user_files/user_15.csv  
  inflating: user_files/user_16.csv  
  inflating: user_files/user_17.csv  
  inflating: user_files/user_18.csv  


## Separate train and test files

In [ ]:
users_events = []
all_events = []
for file in os.listdir(DATA_PATH):
  user_events = pd.read_csv(DATA_PATH + file,  header=None)
  user_events = [[re.sub(r'[^\w\s]','',column_1), re.sub(r'[^\w\s]','',column_2)[1:]] for column_1, column_2 in zip(user_events[0], user_events[1])]
  all_events += user_events
  users_events.append(user_events)

train_files, test_files = [], []
sum = np.array([len(event) for event in users_events]).sum()
curr = 0
unique_events = set()
vocabulary = set()
for event in all_events:
  unique_events.add("->".join(event))
  vocabulary.add(event[0])
  vocabulary.add(event[1])
unique_events_dic = {event:index for index, event in enumerate(unique_events)}
unique_events_label = {index:event for index, event in enumerate(unique_events)}

print("Length of sentences is: ",len(users_events))
train_size = int(sum*0.80)
for user_events in users_events:
  if curr < train_size:
    curr += len(user_events)
    train_files.append(user_events)
  else:
    test_files.append(user_events)
print("Length of train files and test files are: ",len(train_files)," and ", len(test_files),"\n")



Length of sentences is:  58
Length of train files and test files are:  45  and  13 



In [ ]:
for l in range(len(train_files)):
  #if not train_files[l][len(train_files[l])-1]:
  print(len(train_files[l]))
  print(train_files[l][len(train_files[l])-1])

636
['external_storage', 'media_scanner_started']
1880
['external_storage', 'media_scanner_started']
4607
['audio', 'audio_becoming_noisy']
3700
['configuration', 'changed']
1309
['configuration', 'changed']
4285
['ringer', 'ringer_normal']
1214
['configuration', 'changed']
20209
['time_date', 'date_changed']
13448
['configuration', 'changed']
5629
['data_connection', 'wifi_connected']
2420
['time_date', 'next_alarm_changed']
1674
['power', 'power_connected']
588
['data_connection', 'lte_connected']
524
['ringer', 'ringer_normal']
305
['time_date', 'next_alarm_changed']
9312
['time_date', 'next_alarm_changed']
4417
['ringer', 'ringer_vibrate']
151
['ringer', 'ringer_vibrate']
6360
['time_date', 'next_alarm_changed']
126
['time_date', 'date_changed']
8453
['data_connection', 'lte_connected']
910
['time_date', 'next_alarm_changed']
7445
['data_connection', 'wifi_connected']
7237
['time_date', 'date_changed']
1231
['power', 'power_connected']
366
['power', 'power_disconnected']
1432
['dat

In [ ]:
emb_model = Word2Vec(all_events, size = 100, min_count = 1)
word_vec = emb_model.wv
print("word vec is:",len(emb_model.wv.vocab))
embedding_matrix = np.zeros((len(emb_model.wv.vocab), 100))
for i in range(len(emb_model.wv.vocab)):
    embedding_vector =  emb_model.wv[emb_model.wv.index2word[i]]
    if embedding_vector is not None:
      embedding_matrix[i] = embedding_vector

print("Len of emb matrix: ",len(embedding_matrix),len(embedding_matrix[0]))

word vec is: 52
Len of emb matrix:  52 100


In [ ]:
X_train, y_train, z_train = [], [], []
for user_events in train_files:
  for index in range(len(user_events)-1):
    # X_train.append(np.average(emb_model.wv[user_events[index]], axis=0)) # + "->"+user_events[index][1])
    # y_train.append(np.average(emb_model.wv[user_events[index+1]], axis=0)) # + "->"+user_events[index+1][1])
    X_train.append(user_events[index][0]+"."+user_events[index][1])
    y_train.append(user_events[index+1][0]+"."+user_events[index+1][1])
    #z_train.append(user_events[index+2])
print("length of Xtrain and ytrain is: ",len(X_train),len(y_train))

X_test, y_test, z_test = [], [], []
for user_events in test_files:
  for index in range(len(user_events)-1):
    # X_test.append(np.average(emb_model.wv[user_events[index]], axis=0)) # + "->"+user_events[index][1])
    # y_test.append(np.average(emb_model.wv[user_events[index+1]], axis=0)) # + "->"+user_events[index+1][1])
    #if user_events[index][0]+"."+user_events[index][1] and user_events[index+1][0]+"."+user_events[index+1][1] in X_train:
    X_test.append(user_events[index][0]+"."+user_events[index][1])
    y_test.append(user_events[index+1][0]+"."+user_events[index+1][1])
    #z_test.append(user_events[index+2])

print("length of Xtest and ytest is: ",len(X_test),len(y_test))
#X_train, y_train, X_test, y_test = np.array(X_train), np.array(y_train) , np.array(X_test), np.array(y_test)

length of Xtrain and ytrain is:  141140 141140
length of Xtest and ytest is:  29661 29661


# Model

In [ ]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crf.fit([X_train], [y_train], X_dev=None,y_dev=None)

/usr/local/lib/python3.7/dist-packages/sklearn/base.py:213: FutureWarning: From version 0.24, get_params will raise an AttributeError if a parameter cannot be retrieved as an instance attribute. Previously it would return None.
  FutureWarning)


CRF(algorithm='lbfgs', all_possible_transitions=True, c1=0.1, c2=0.1,
    keep_tempfiles=None, max_iterations=100)

In [ ]:
latent_crf = LatentNodeCRF(n_labels=51, n_features=1, n_hidden_states=2,
                           inference_method='lp')
ssvm = OneSlackSSVM(model=latent_crf, max_iter=200, C=100,
                    n_jobs=-1, show_loss_every=10, inference_cache=50)
latent_svm = LatentSSVM(ssvm)
latent_svm.fit(X_train, y_test)

In [ ]:
  labels = list(crf.classes_)

In [ ]:
len(labels)

38

In [ ]:
y_pred = crf.predict([X_test])
metrics.flat_f1_score([y_test], y_pred,
                      average='weighted', labels=labels)

/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1465: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  average, "true nor predicted", 'F-score is', len(true_sum)


0.5822049475327978

In [ ]:
sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)
print(metrics.flat_classification_report(
    [y_test], y_pred, labels=sorted_labels, digits=3
))

/usr/local/lib/python3.7/dist-packages/sklearn/utils/validation.py:70: FutureWarning: Pass labels=['data_connection.airplane_mode', 'data_connection.edge_connected', 'data_connection.hsdpa_connected', 'data_connection.hspa_connected', 'data_connection.lte_connected', 'data_connection.umts_connected', 'data_connection.wifi_connected', 'headset.headset_plugged', 'headset.headset_unplugged', 'wifi_device.off', 'wifi_device.on', 'time_date.date_changed', 'time_date.next_alarm_changed', 'time_date.time_set', 'time_date.timezone_changed', 'ringer.ringer_normal', 'ringer.ringer_silent', 'ringer.ringer_vibrate', 'misc.external_apps_unavailable', 'misc.input_method_changed', 'misc.wallpaper_changed', 'location.gps_activity', 'configuration.changed', 'power.battery_low', 'power.battery_ok', 'power.power_connected', 'power.power_disconnected', 'usb.device_attached', 'usb.device_detached', 'audio.audio_becoming_noisy', 'audio.audio_effects_closed', 'audio.audio_effects_opened', 'external_storage.b

                                          precision    recall  f1-score   support

           data_connection.airplane_mode      0.000     0.000     0.000        36
          data_connection.edge_connected      0.000     0.000     0.000        11
         data_connection.hsdpa_connected      0.000     0.000     0.000         0
          data_connection.hspa_connected      0.500     0.011     0.021        95
           data_connection.lte_connected      0.797     0.743     0.769      7249
          data_connection.umts_connected      0.000     0.000     0.000         0
          data_connection.wifi_connected      0.674     0.849     0.752      5689
                 headset.headset_plugged      0.189     0.131     0.155       482
               headset.headset_unplugged      0.577     0.569     0.573       557
                         wifi_device.off      0.544     0.383     0.450       256
                          wifi_device.on      0.708     0.796     0.749       402
               

## Hyperparameter optimization

In [ ]:
%%time
# define fixed parameters and parameters to search
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=100,
    all_possible_transitions=True
)
params_space = {
    'c1': scipy.stats.expon(scale=0.5),
    'c2': scipy.stats.expon(scale=0.05),
}

# use the same metric for evaluation
f1_scorer = make_scorer(metrics.flat_f1_score,
                        average='weighted', labels=labels)

# search
rs = RandomizedSearchCV(crf, params_space,
                        cv=3,
                        verbose=1,
                        n_jobs=-1,
                        n_iter=50,
                        scoring=f1_scorer)
rs.fit(X_train, y_train)


In [ ]:
# crf = rs.best_estimator_
print('best params:', rs.best_params_)
print('best CV score:', rs.best_score_)
print('model size: {:0.2f}M'.format(rs.best_estimator_.size_ / 1000000))



## Check parameter space

In [ ]:
_x = [s.parameters['c1'] for s in rs.scorer_]
_y = [s.parameters['c2'] for s in rs.grid_scores_]
_c = [s.mean_validation_score for s in rs.grid_scores_]

fig = plt.figure()
fig.set_size_inches(12, 12)
ax = plt.gca()
ax.set_yscale('log')
ax.set_xscale('log')
ax.set_xlabel('C1')
ax.set_ylabel('C2')
ax.set_title("Randomized Hyperparameter Search CV Results (min={:0.3}, max={:0.3})".format(
    min(_c), max(_c)
))

ax.scatter(_x, _y, c=_c, s=60, alpha=0.9, edgecolors=[0,0,0])

print("Dark blue => {:0.4}, dark red => {:0.4}".format(min(_c), max(_c)))

## Check best estimator

In [ ]:
crf = rs.best_estimator_
y_pred = crf.predict(X_test)
print(metrics.flat_classification_report(
    y_test, y_pred, labels=sorted_labels, digits=3
))

## Classifier learning

In [ ]:
def print_transitions(trans_features):
    for (label_from, label_to), weight in trans_features:
        print("%s -> %s %f" % (label_from, label_to, weight))

print("Top likely transitions:")
print_transitions(Counter(crf.transition_features_).most_common(100))

print("\nTop unlikely transitions:")
print_transitions(Counter(crf.transition_features_).most_common()[-100:])

Top likely transitions:
external_storage.ejected -> external_storage.unmounted 9.506142
external_storage.media_scanner_started -> external_storage.media_scanner_finished 6.841776
usb.device_detached -> usb.device_attached 5.874298
audio.audio_effects_closed -> audio.audio_effects_opened 5.076873
external_storage.unmounted -> external_storage.bad_removal 4.937101
usb.device_detached -> external_storage.ejected 4.734773
data_connection.edge_connected -> data_connection.umts_connected 4.643469
data_connection.airplane_mode -> data_connection.airplane_mode 4.514259
data_connection.airplane_mode -> usb.device_attached 4.067724
data_connection.edge_connected -> data_connection.edge_connected 3.945678
time_date.timezone_changed -> time_date.timezone_changed 3.905738
usb.device_attached -> headset.headset_plugged 3.656487
audio.audio_becoming_noisy -> headset.headset_unplugged 3.627305
wifi_device.on -> ringer.ringer_normal 3.528859
data_connection.umts_connected -> data_connection.umts_connec

In [ ]:
vals = ['battery_ok']
for val in vals:
  for x in X_train:
    if val in x:
      print(x)
      break

In [ ]:
X_train[:100]

In [ ]:
def print_state_features(state_features):
    for (attr, label), weight in state_features:
        print("%0.6f %-8s %s" % (weight, label, attr))

print("Top positive:")
print_state_features(Counter(crf.state_features_).most_common(30))

print("\nTop negative:")
print_state_features(Counter(crf.state_features_).most_common()[-30:])

